# Local Interpretable Model-agnostic Explanations (LIME) 
## Pneumonia Image Classification

LIME perturbs the inputs to the model and tests how the predictions change. By randomly hiding regions of the X-Ray ("superpixels") and measuring the prediction swing, LIME builds a linear model locally around the prediction to highlight the regions that were most important.



In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from tensorflow.keras.preprocessing import image
from lime import lime_image
from skimage.segmentation import mark_boundaries

print(f"TensorFlow version: {tf.__version__}")

## 1. Load the Trained Model


In [ ]:
MODEL_PATH = 'best_pneumonia_model.keras'

if not os.path.exists(MODEL_PATH):
    print("Error: Model not found. Run training notebook first!")
else:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("Pneumonia Classification Model loaded successfully.")

## 2. Setup the LIME Explainer & Prediction Wrapper
LIME requires a prediction function that accepts batches of raw images and returns a 2D array of class probabilities summing to 1. Since our model is binary (1 sigmoid output node), we manually create a two-class probability distribution `[1-p, p]`.


In [ ]:
# We need to preprocess images before they hit the ResNet/DenseNet layers
base_model_index = -1
for i, layer in enumerate(model.layers):
    if 'densenet' in layer.name.lower() or (isinstance(layer, tf.keras.Model) and not isinstance(layer, tf.keras.Sequential)):
        base_model_index = i
        break

# Slice the official preprocessing layers out
preprocess_model = tf.keras.Model(inputs=model.inputs, outputs=model.layers[base_model_index - 1].output)

def predict_fn(images):
    # LIME passes floating-point images [0, 1] internally for skimage if scaled, but we feed it [0, 255]
    # We enforce raw uint8/float32 compatibility based on what we feed it. 
    # Our explainer will receive the 0-255 image.
    
    # Process images (data augmentation is bypassed by training=False)
    processed_images = preprocess_model(images, training=False)
    
    # Get model probabilities (shape is batch x 1)
    preds = model.predict(processed_images, verbose=0)
    
    # LIME needs shape (batch, num_classes), so we reconstruct the 0th class (Normal) and 1st class (Pneumonia)
    # [ P(Normal), P(Pneumonia) ]
    lime_preds = np.hstack((1 - preds, preds))
    return lime_preds

# Initialize the image explainer
explainer = lime_image.LimeImageExplainer()

## 3. Explanation Visualization Function


In [ ]:
def explain_and_visualize(img_path, bbox=None, num_samples=300):
    if not os.path.exists(img_path):
        print(f"Image not found: {img_path}")
        return
        
    print(f"\nExplaining {img_path} with LIME...")
    # Load raw image directly
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    
    # Convert exactly to the format preprocess_model expects (float32 numpy array, 0-255)
    img_array = img_array.astype(np.float32)
    
    # Generate explanation for class 1 (Pneumonia)
    # Note: num_samples dictates how many perturbed images LIME generates (computation time)
    explanation = explainer.explain_instance(
        img_array, 
        predict_fn, 
        top_labels=2, 
        hide_color=0, 
        num_samples=num_samples
    )
    
    # Extract image and mask mapped to Pneumonia's positive attribution
    temp, mask = explanation.get_image_and_mask(
        label=1, 
        positive_only=True, 
        num_features=5, 
        hide_rest=False
    )
    
    # temp is returned normalized to [0, 1] internally by LIME wrapper (sometimes)
    # Convert safely back for pyplot
    if temp.max() <= 1.0:
        temp = np.uint8(temp * 255.0)
    else:
        temp = np.uint8(temp)
        
    img_boundry = mark_boundaries(temp, mask)
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    
    ax[0].set_title("Original X-Ray (Ground Truth BBox)")
    ax[0].imshow(image.load_img(img_path), cmap='gray')
    
    if bbox:
        # Scale bbox to 224x224 from original 1024x1024 dataset size
        original_img_size = 1024.0
        current_img_size = 224.0
        scale_factor = current_img_size / original_img_size
        scaled_bbox = [val * scale_factor for val in bbox]
        
        rect = patches.Rectangle((scaled_bbox[0], scaled_bbox[1]), scaled_bbox[2], scaled_bbox[3], linewidth=2, edgecolor='r', facecolor='none', label='Ground Truth')
        ax[0].add_patch(rect)
        rect2 = patches.Rectangle((scaled_bbox[0], scaled_bbox[1]), scaled_bbox[2], scaled_bbox[3], linewidth=2, edgecolor='white', facecolor='none', label='Ground Truth')
        ax[1].add_patch(rect2)
        ax[0].legend()
        ax[1].legend()

    ax[0].axis("off")
    
    ax[1].set_title("LIME Explanation (Positive Pneumonia Superpixels)")
    ax[1].imshow(img_boundry)
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 4. Evaluate Explanations
LIME runs hundreds of inferences, so it takes a little longer than Grad-CAM but gives highly granular superpixel insights.


In [ ]:
# 4.1 Test on the specific BBox image
bbox_img_path = "C:/Users/prakh/Downloads/archive (1)/chest_xray/00011136_002.png"
gt_bbox = [574.00, 574.90, 229.83, 166.11] 
explain_and_visualize(bbox_img_path, bbox=gt_bbox, num_samples=300)

# 4.2 Test on a regular test image
test_pneumonia_dir = "C:/Users/prakh/Downloads/archive (1)/chest_xray/test/PNEUMONIA"
if os.path.exists(test_pneumonia_dir):
    sample_img_name = os.listdir(test_pneumonia_dir)[0]
    img_path = os.path.join(test_pneumonia_dir, sample_img_name)
    explain_and_visualize(img_path, num_samples=300)
